In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Importation des données

In [2]:
df = pd.read_csv("../data/churn_data.csv")
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

## Analyse

Premier problème que j'observe, `TotalCharges` devrait être une colonne numérique. Mais ce n'est pas le cas il doit sûrement y avoir des chiffres qui sont sous format `str`

In [4]:
print(df.isna().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


Il n'y en plus de ça aucune valeur manquante.

In [5]:
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True))

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


Actuellement avec les données un peu plus de 26% des clients ont churn.
Ça veut donc dire que les données sont légèrement déséquilibrés et que seule **l'accuracy** ne pourra pas être utilisé comme metrics de performance. Sans quoi le modèle pourrais prédire constamment **No** et ***l'accuracy*** serait déjà à 73.5%

In [6]:
# On va convertir toutes les données en numériques et celles qui bloqueront seront renvoyés comme NaN
totalcharges_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")

lignes_problematiques = df[totalcharges_numeric.isna()]
print(f"Nombre de ligne problématiques : {len(lignes_problematiques)}")
lignes_problematiques[["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

Nombre de ligne problématiques : 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


On a donc 11 lignes qui posent problèmes, mais il est facilement identifiables que ce sont pour des clients qui n'ont pas encore été facturé, donc cela est plus logique.

In [7]:
print(df.loc[lignes_problematiques.index, "Churn"].value_counts())

Churn
No    11
Name: count, dtype: int64


Au final on voit qu'aucun client n'a churné parmis ceux qui n'ont encore aucune facture cumulé.

In [8]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [9]:
print("Type après nettoyage : ", df["TotalCharges"].dtype)
print("NaN restants : ", df["TotalCharges"].isna().sum())

Type après nettoyage :  float64
NaN restants :  0


Nettoyage terminé pour la colonne `df[TotalCharges]`

### Analyse et étude variables catégorielles

In [10]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
cat_cols = [col for col in cat_cols if col not in ["customerID", "Churn"]]

for col in cat_cols:
    print(f"---{col}---")
    print(df[col].value_counts(), "\n")


---gender---
gender
Male      3555
Female    3488
Name: count, dtype: int64 

---Partner---
Partner
No     3641
Yes    3402
Name: count, dtype: int64 

---Dependents---
Dependents
No     4933
Yes    2110
Name: count, dtype: int64 

---PhoneService---
PhoneService
Yes    6361
No      682
Name: count, dtype: int64 

---MultipleLines---
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64 

---InternetService---
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64 

---OnlineSecurity---
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64 

---OnlineBackup---
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64 

---DeviceProtection---
DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name:

/var/folders/cy/9f5cj2px2nb99gn8c9v2qzf80000gn/T/ipykernel_36930/18257151.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns.tolist()


Dans **6 colonnes** au total la mention `No internet service` apparaît 1526 fois.
En parallèle 1526, c'est le nombre de client qui ont `InternetService = No`  

Même remarque pour `No Phone Service` dans la colonne `MultipleLines` avec 682 qui est l'équivalent de `PhoneService = No`, selon ma première observation.

Pour éviter de créer des colonnes supplémentaires avec le OHE, qui pourraient prendre du poids dans le modèle, il semblerait bon de fusionner les différentes mentions en un simple `No`. Cela évitera 

In [12]:
cols_internet = ["OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]

for col in cols_internet:
    df[col] = df[col].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

In [13]:
for col in cols_internet + ["MultipleLines"]:
    print(f"---{col}---")
    print(df[col].value_counts(), "\n")

---OnlineSecurity---
OnlineSecurity
No     5024
Yes    2019
Name: count, dtype: int64 

---OnlineBackup---
OnlineBackup
No     4614
Yes    2429
Name: count, dtype: int64 

---DeviceProtection---
DeviceProtection
No     4621
Yes    2422
Name: count, dtype: int64 

---TechSupport---
TechSupport
No     4999
Yes    2044
Name: count, dtype: int64 

---StreamingTV---
StreamingTV
No     4336
Yes    2707
Name: count, dtype: int64 

---StreamingMovies---
StreamingMovies
No     4311
Yes    2732
Name: count, dtype: int64 

---MultipleLines---
MultipleLines
No     4072
Yes    2971
Name: count, dtype: int64 



***Problème réglé***

### Étude ciblé par colonne

In [18]:
print("---Churn par contrat---")
print(pd.crosstab(df["Contract"], df["Churn"], normalize="index"))

print("\n---Churn par InternetService---")
print(pd.crosstab(df["InternetService"], df["Churn"], normalize="index"))

print("\n---Tenure moyenne selon Churn---")
print(df.groupby("Churn")["tenure"].mean())

print("\n---MonthlyCharges moyenne selon Churn---")
print(df.groupby("Churn")["MonthlyCharges"].mean())

---Churn par contrat---
Churn                 No       Yes
Contract                          
Month-to-month  0.572903  0.427097
One year        0.887305  0.112695
Two year        0.971681  0.028319

---Churn par InternetService---
Churn                  No       Yes
InternetService                    
DSL              0.810409  0.189591
Fiber optic      0.581072  0.418928
No               0.925950  0.074050

---Tenure moyenne selon Churn---
Churn
No     37.569965
Yes    17.979133
Name: tenure, dtype: float64

---MonthlyCharges moyenne selon Churn---
Churn
No     61.265124
Yes    74.441332
Name: MonthlyCharges, dtype: float64


On observe que le churn pas contrat est le plus récurrent quand on est sur une typologie de contrat `Month-to-month`. Ensuite, la `Fiber optic` le service internet où on observe le plus gros ratio de churn.

Aussi les clients qui churns on en moyenne 18 mois d'ancienneté ce qui paraît logique pour des contrats qui pourrait être souscripts pour une période d'un an et demi.
Et en moyenne les charges mensuelles chez les clients qui churn on des `MonthlyCharges` plus élevé que les autres clients qui ne chrunent pas.